In [16]:
from google.colab import drive
drive.mount('/content/drive')

import sys
import os
import json
from pathlib import Path

import pandas as pd

import numpy as np

import psutil
from tqdm import tqdm

import kagglehub

import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW

from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    get_linear_schedule_with_warmup
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path(os.path.abspath('')).parent.parent
sys.path.append(str(PROJECT_ROOT))

from core.cleaning import clean_text, SLANG_DICT
from core.data_loaders import load_pan13, load_bf_psr, load_pan12
from core.training import ChatDataset, train_epoch, evaluate, save_model

randmansour_pan_13_path = kagglehub.dataset_download('randmansour/pan-13')
pan12_data_path = kagglehub.dataset_download('randmansour/pan12-data')
bf_psr_framework_path = kagglehub.dataset_download('randmansour/bf-psr-framework')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using Colab cache for faster access to the 'pan-13' dataset.
Using Colab cache for faster access to the 'pan12-data' dataset.
Using Colab cache for faster access to the 'bf-psr-framework' dataset.


## Loading Pan12 and PJZC

In [17]:
memory = psutil.virtual_memory()
print(f"Total RAM: {memory.total / 1e9:.2f} GB")
print(f"Available RAM: {memory.available / 1e9:.2f} GB")
print(f"RAM Usage: {memory.percent}%")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

PAN12_PATHS = {
 'train_xml': '/kaggle/input/pan12-data/pan12-training-corpus-2012-05-01/pan12-sexual-predator-identification-training-corpus-2012-05-01/pan12-sexual-predator-identification-training-corpus-2012-05-01.xml',
 'train_ids': '/kaggle/input/pan12-data/pan12-training-corpus-2012-05-01/pan12-sexual-predator-identification-training-corpus-2012-05-01/pan12-sexual-predator-identification-training-corpus-predators-2012-05-01.txt',
 'test_xml': '/kaggle/input/pan12-data/pan12-test-corpus-2012-05-21/pan12-sexual-predator-identification-test-corpus-2012-05-21/pan12-sexual-predator-identification-test-corpus-2012-05-17.xml',
 'test_ids': '/kaggle/input/pan12-data/pan12-test-corpus-2012-05-21/pan12-sexual-predator-identification-test-corpus-2012-05-21/pan12-sexual-predator-identification-groundtruth-problem1.txt'
}


pan12_train = load_pan12(PAN12_PATHS['train_xml'], PAN12_PATHS['train_ids'])
pan12_test = load_pan12(PAN12_PATHS['test_xml'], PAN12_PATHS['test_ids'])

print(f"\nPAN12 Training: {len(pan12_train)} messages, {pan12_train['conversation_id'].nunique()} conversations")
print(f"PAN12 Test: {len(pan12_test)} messages, {pan12_test['conversation_id'].nunique()} conversations")

PJZC = load_bf_psr('/kaggle/input/bf-psr-framework/JsonData/PJZC.txt')

print(f"\nPJZC: {len(PJZC)} messages, {PJZC['conversation_id'].nunique()} conversations")


Total RAM: 13.61 GB
Available RAM: 4.76 GB
RAM Usage: 65.0%

PAN12 Training: 903607 messages, 66927 conversations
PAN12 Test: 2058781 messages, 155128 conversations

PJZC: 372161 messages, 21070 conversations


In [18]:
PAN13_PATHS = {
 'training': {
 'xml_dir': '/root/.cache/kagglehub/datasets/randmansour/pan-13/versions/1/pan13-author-profiling-training-corpus-2013-01-09/pan13-author-profiling-training-corpus-2013-01-09/en',
 'truth_file': '/root/.cache/kagglehub/datasets/randmansour/pan-13/versions/1/pan13-author-profiling-training-corpus-2013-01-09/pan13-author-profiling-training-corpus-2013-01-09/truth-en.txt'
    },
 'test': {
 'xml_dir': '/root/.cache/kagglehub/datasets/randmansour/pan-13/versions/1/pan13-author-profiling-test-corpus2-2013-04-29/pan13-author-profiling-test-corpus2-2013-04-29/en',
 'truth_file': '/root/.cache/kagglehub/datasets/randmansour/pan-13/versions/1/pan13-author-profiling-test-corpus2-2013-04-29/pan13-author-profiling-test-corpus2-2013-04-29/truth-en.txt'
    }
}

pan13_train = load_pan13(
    PAN13_PATHS['training']['xml_dir'],
    dataset_type='pan13_train',
    truth_file=PAN13_PATHS['training']['truth_file']
)

pan13_test = load_pan13(
    PAN13_PATHS['test']['xml_dir'],
    dataset_type='pan13_test',
    truth_file=PAN13_PATHS['test']['truth_file']
)

print(f"\nPAN13 Training: {len(pan13_train)} documents, {pan13_train['author_id'].nunique() if len(pan13_train) > 0 else 0} authors")
print(f"PAN13 Test: {len(pan13_test)} documents, {pan13_test['author_id'].nunique() if len(pan13_test) > 0 else 0} authors")


Loading PAN13 truth file: /root/.cache/kagglehub/datasets/randmansour/pan-13/versions/1/pan13-author-profiling-training-corpus-2013-01-09/pan13-author-profiling-training-corpus-2013-01-09/truth-en.txt
Loaded 236600 author mappings from truth file
Processed 236600 XML files
Matched 236600 authors to truth file (100.0%)
Created 413555 conversation rows
Loading PAN13 truth file: /root/.cache/kagglehub/datasets/randmansour/pan-13/versions/1/pan13-author-profiling-test-corpus2-2013-04-29/pan13-author-profiling-test-corpus2-2013-04-29/truth-en.txt
Loaded 25440 author mappings from truth file
Processed 25440 XML files
Matched 25440 authors to truth file (100.0%)
Created 47927 conversation rows

PAN13 Training: 413555 documents, 236600 authors
PAN13 Test: 47927 documents, 25440 authors


In [19]:
pan13_train=pan13_train[pan13_train['category'].notnull()][['author_id'	,'doc_id'	,'text'	]]
pan13_train.rename(columns={'author_id': 'author', 'doc_id': 'conversation_id'}, inplace=True)
pan13_train['source']='PAN13_TRAIN'
pan13_test=pan13_test[pan13_test['category'].notnull()][['author_id'	,'doc_id'	,'text'	]]
pan13_test.rename(columns={'author_id': 'author', 'doc_id': 'conversation_id'}, inplace=True)
pan13_test['source']='PAN13_TEST'
pan13 = pd.concat([pan13_train, pan13_test], ignore_index=True)

In [20]:
# Group PAN13 messages by author to create longer contexts
pan13_by_author = pan13.groupby('author').agg({
 'text': ' '.join,
 'source': 'first'
}).reset_index()

# Create conversation_id from author
pan13_by_author['conversation_id'] = pan13_by_author['author']
pan13_by_author['label']=1


PAN12 Sexual Predator Identification Dataset: 66,004

PJZC Dataset: 19,966

Because PJZC is newer, we want to give it strong representation, but PAN12 still has more structured conversations.

A good balanced split is 60% PAN12 / 40% PJZC.

Normal total = 18,720


In [21]:
# 1. First combine all messages into conversations
def combine_messages(df):
    return df.groupby('conversation_id').agg({
 'text': ' '.join,
 'label': 'max',
 'source': 'first'
    }).reset_index()


predatory_pan12 = pan12_train[pan12_train['label'] == 1]
predatory_pjzc = PJZC[PJZC['label'] == 1]
predatory_df = pd.concat([predatory_pan12, predatory_pjzc], ignore_index=True)

# Normal conversations - keep all PJZC normal, subsample PAN12 normal
normal_pjzc = PJZC[PJZC['label'] == 0]
normal_pan12 = pan12_train[pan12_train['label'] == 0]
normal_pan12_combined = combine_messages(normal_pan12)
normal_pjzc_combined = combine_messages(normal_pjzc)

predatory_df_combined = combine_messages(predatory_df)
predatory_df= pd.concat([predatory_df_combined, pan13_by_author], ignore_index=True)

pan12_sample = normal_pan12_combined.sample(
    n=min(12200, len(normal_pan12_combined)),
    random_state=42
)

pjzc_sample = normal_pjzc_combined.sample(
    n=min(8134, len(normal_pjzc_combined)),
    random_state=42
)

normal_df = pd.concat([pan12_sample, pjzc_sample], ignore_index=True)

print(f"Normal conversations: {len(normal_df)}")
print(f"Pred conversations: {len(predatory_df)}")
print(f"Normal to pred ratio {len(normal_df)/len(predatory_df)}:1")


Normal conversations: 20334
Pred conversations: 3389
Normal to pred ratio 6.0:1


In [22]:
# 1. Combine all conversations
all_conversations = pd.concat([predatory_df_combined, normal_df], ignore_index=True)

# 2. Clean text
tqdm.pandas(desc="Cleaning all conversations")
all_conversations['text_clean'] = all_conversations['text'].progress_apply(
    lambda x: clean_text(x, aggressive=True, slang_dict=SLANG_DICT)
)

# 3. Remove empty conversations
all_conversations_clean = all_conversations[
    all_conversations['text_clean'].str.len() > 0
].reset_index(drop=True)

print(f"Removed {len(all_conversations) - len(all_conversations_clean)} empty conversations")

# 4. Compute conversation length (words)
all_conversations_clean['length'] = all_conversations_clean['text_clean'].str.split().str.len()

# 5. Remove very short conversations (recommended < 10 words)
MIN_WORDS = 10

before = len(all_conversations_clean)

all_conversations_clean = all_conversations_clean[
    all_conversations_clean['length'] >= MIN_WORDS
].reset_index(drop=True)

print(f"Removed {before - len(all_conversations_clean)} short conversations (<{MIN_WORDS} words)")


# 1. Remove duplicates based on cleaned text
before_duplicates = len(all_conversations_clean)
all_conversations_clean = all_conversations_clean.drop_duplicates(subset=['text_clean'])
print(f" Removed {before_duplicates - len(all_conversations_clean)} duplicate conversations")

# 2. Check for and remove nulls
print(f"\nNull values before:")
print(all_conversations_clean[['text_clean', 'label']].isnull().sum())

all_conversations_clean = all_conversations_clean.dropna(subset=['text_clean', 'label'])
print(f"\n Removed null rows")


# 5. Final check
print(f"Total samples after all cleaning: {len(all_conversations_clean)}")
print(f"Duplicates: {all_conversations_clean.duplicated(subset=['text_clean']).sum()}")
print(f"Nulls in text_clean: {all_conversations_clean['text_clean'].isnull().sum()}")
print(f"Nulls in label: {all_conversations_clean['label'].isnull().sum()}")
print(f"\nLabel distribution:")
print(all_conversations_clean['label'].value_counts())
# 6. Train / Validation / Test split
conversations = all_conversations_clean[['conversation_id', 'label']].drop_duplicates()

train_conv, temp_conv = train_test_split(
    conversations,
    test_size=0.2,
    stratify=conversations['label'],
    random_state=42
)

valid_conv, test_conv = train_test_split(
    temp_conv,
    test_size=0.5,
    stratify=temp_conv['label'],
    random_state=42
)

# 7. Map back to full dataframe
train_df = all_conversations_clean[
    all_conversations_clean['conversation_id'].isin(train_conv['conversation_id'])
]

valid_df = all_conversations_clean[
    all_conversations_clean['conversation_id'].isin(valid_conv['conversation_id'])
]

test_df = all_conversations_clean[
    all_conversations_clean['conversation_id'].isin(test_conv['conversation_id'])
]

# 8. Check distributions
print("\nFinal distributions:")
print(f"Train: {train_df['label'].value_counts().to_dict()}")
print(f"Valid: {valid_df['label'].value_counts().to_dict()}")
print(f"Test: {test_df['label'].value_counts().to_dict()}")

Cleaning all conversations: 100%|██████████| 23454/23454 [00:38<00:00, 607.82it/s]


Removed 41 empty conversations
Removed 7236 short conversations (<10 words)
Removed 79 duplicate conversations

Null values before:
text_clean    0
label         0
dtype: int64

Removed null rows

Total samples after all cleaning: 16098
Duplicates: 0
Nulls in text_clean: 0
Nulls in label: 0

Label distribution:
label
0    13965
1     2133
Name: count, dtype: int64

Final distributions:
Train: {0: 11201, 1: 1735}
Valid: {0: 1416, 1: 229}
Test: {0: 1413, 1: 234}


In [23]:
print(f"Train Normal to pred ratio {train_df['label'].value_counts().to_dict()[0]/train_df['label'].value_counts().to_dict()[1]}:1")
print(f"Valid Normal to pred ratio {valid_df['label'].value_counts().to_dict()[0]/valid_df['label'].value_counts().to_dict()[1]}:1")
print(f"Test Normal to pred ratio {test_df['label'].value_counts().to_dict()[0]/test_df['label'].value_counts().to_dict()[1]}:1")

Train Normal to pred ratio 6.455907780979827:1
Valid Normal to pred ratio 6.183406113537118:1
Test Normal to pred ratio 6.038461538461538:1


In [24]:

# Show sample comparisons
for idx in range(min(5, len(train_df))):
    print("\n" + "-"*80)
    print(f"SAMPLE {idx+1} (Label: {train_df['label'].iloc[idx]})")
    print("-"*80)

    # Get original text (we need to reload it since we overwrote it)
    conv_id = train_df['conversation_id'].iloc[idx]
    original_text = all_conversations[all_conversations['conversation_id'] == conv_id]['text'].iloc[0]

    print("\n RAW TEXT:")
    print(original_text[:500] + "..." if len(original_text) > 500 else original_text)

    print("\n CLEANED TEXT:")
    cleaned = train_df['text_clean'].iloc[idx]  # Fixed: using text_clean
    print(cleaned[:500] + "..." if len(cleaned) > 500 else cleaned)



--------------------------------------------------------------------------------
SAMPLE 1 (Label: 1)
--------------------------------------------------------------------------------

RAW TEXT:
hi how r ya? was out with a buddy u miss me huh? so is that a new pic? cool wish u had a hot pic 2 c ;) its ok that was a nice pic so how u getting to b up late? oh my u rock sweety yeah baby u gonna rock my world? hehe is that gonna b a good time? yep as long as u feel good with it yeah so how should we start any ideas? yes hmm battery powered toy? u like oils? what else u might like? oh i dont mind so r u shy? was just going to see if u want me 2 put a move on u or u put one on me? if u r cool...

CLEANED TEXT:
hi how are ya? was out with a buddy you miss me huh? so is that a new picture? cool wish you had a hot picture 2 see; ) it is ok that was a nice picture so how you getting to be up late? oh my you rock sweety yeah baby you going to rock my world? hehe is that going to be a good time? ye

In [ ]:
DATA_PATH = os.getenv('DATA_PATH')
os.makedirs(DATA_PATH, exist_ok=True)

print("\n Training label distribution (after cleaning):")
print(train_df['label'].value_counts())
print(f"\n  Predators: {sum(train_df['label'] == 1)}")
print(f" Non-predators: {sum(train_df['label'] == 0)}")

# Save the FULL cleaned dataset to Drive
train_file = f'{DATA_PATH}/predator_train_cleaned.csv'
train_df.to_csv(train_file, index=False)
print(f" Saved FULL cleaned dataset to {train_file}")
print(f" Shape: {train_df.shape}")

# Save validation data to Drive
valid_file = f'{DATA_PATH}/predator_valid_cleaned.csv'
valid_df.to_csv(valid_file, index=False)
print(f" Saved cleaned validation data to {valid_file}")
print(f" Shape: {valid_df.shape}")

# Save test data to Drive
test_file = f'{DATA_PATH}/predator_test_cleaned.csv'
test_df.to_csv(test_file, index=False)
print(f" Saved cleaned test data to {test_file}")
print(f" Shape: {test_df.shape}")

In [26]:
_

''